# Lab 11: Chain-of-Thought

## Making the model show its work

The next cells build `build_direct_prompt` (asks for a short answer immediately) and `build_cot_prompt` (asks the model to reason step by step, then end with a `"Answer: <answer>"` line), plus `extract_final_answer` to pull that final line back out. See the Reading tab for what a reasoning trace is and why comparing a direct answer against a step-by-step one matters.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


### Concept 1 practice: comparing two approaches side by side
A quick blurt-it-out vs. show-your-work comparison on a different calculation.

In [ ]:
def guess_total(price, tax_rate):
    """Straight to an answer - fast, but nothing to check if it's wrong."""
    return price + price * tax_rate

def show_total_steps(price, tax_rate):
    """Break it into steps you can each verify."""
    tax = price * tax_rate
    total = price + tax
    return f"tax={tax:.2f}, total={total:.2f}"

print(guess_total(50, 0.08))        # expected output: 54.0
print(show_total_steps(50, 0.08))   # expected output: tax=4.00, total=54.00

### Concept 2 practice: pulling a value out of a larger string
The same "split on the last marker" trick, this time pulling a value out of a weather forecast.

In [ ]:
forecast = "Morning: cloudy\nAfternoon: sunny\nHigh: 78F"

def extract_high(text: str, marker: str = "High:") -> str:
    if marker not in text:
        return text.strip()
    return text.rsplit(marker, 1)[1].strip()

print(extract_high(forecast))  # expected output: 78F

## Exercise

### Goal

Sometimes a model answers a hard question better if it is asked to reason out loud first. This is called chain-of-thought prompting. In this exercise you write three small helpers: one prompt that asks for a quick answer, one prompt that asks for step-by-step reasoning ending in a marked final answer, and one function that picks the final answer out of a long reply. You do not call any API here; you only build and read plain Python values.

### Words you will see

- **Message**: one chat turn, written as a dictionary such as `{"role": "user", "content": "Hello"}`. A prompt here is a list holding one such message.
- **Direct prompt**: the question plus a request for a short answer only.
- **Chain of thought (CoT)**: asking the model to "think step by step" so it writes its working before the answer.
- **Marker**: a fixed label such as `Answer:` that you ask the model to put before its final answer, so your code can find it reliably.
- **Default argument**: `marker: str = "Answer:"` means that if the caller does not pass a marker, `"Answer:"` is used.
- **Strip**: remove spaces and newlines from both ends of a string, with `.strip()`.

### `build_direct_prompt(question)`

**Receives:** `question`, a string. Example: `"What is 12 * 13?"`.

**Returns:** a list holding exactly one user message. Example (your wording may differ):

```
[{"role": "user", "content": "What is 12 * 13? Answer in as few words as possible."}]
```

**The idea:** put the question and a short instruction such as "answer in as few words as possible" into one string, and wrap it in a one-item list. The model is not invited to show its working.

**Check:** `len(build_direct_prompt("Hi"))` is `1`, the item's `"role"` is `"user"`, and its `"content"` contains `"Hi"`.

### `build_cot_prompt(question)`

**Receives:** `question`, a string, such as `"What is 12 * 13?"`.

**Returns:** a list holding exactly one user message. The content must contain the question and must tell the model to think **step by step** and to finish with a last line of the form `Answer: <final answer>`.

**The idea:** the same one-message list as before, but the instruction asks for reasoning first and then a clearly marked last line. The marker is what lets `extract_final_answer` find the result later. The checks look for the exact phrase `step by step` (in any capital letters), so make sure it appears in your instruction.

**Check:** `"step by step" in build_cot_prompt("What is 12 * 13?")[0]["content"].lower()` is `True`, and the content also contains `"Answer:"`.

### `extract_final_answer(reply_text, marker="Answer:")`

**Receives:**

- `reply_text`: the model's whole reply as a string, which may be many lines long.
- `marker`: the label to look for. It defaults to `"Answer:"`.

**Returns:** a string, namely everything after the **last** occurrence of the marker, with spaces and newlines stripped from both ends. If the marker is not in the reply, it returns the whole reply stripped.

Worked examples:

- `"Let's see... 12*13=156.\nAnswer: 156"` returns `"156"`.
- `"Answer: 1\nAnswer: 2"` returns `"2"` (the last marker wins).
- `"Answer:   42  \n"` returns `"42"` (spaces and the newline are removed).
- `"  no marker  "` returns `"no marker"`.
- `extract_final_answer("Reasoning\nFinal: 7", "Final:")` returns `"7"` (a custom marker).

Matching is exact, so `"answer:"` in lowercase does not match `"Answer:"`.

**The idea:** first test whether the marker is in the text with `in`. If not, return the stripped text. If it is, split the text at the last marker (the string method `rsplit` with a limit of 1 splits from the right) and keep the piece after it, stripped.

**Check:** `extract_final_answer("so... Answer: yes")` is `"yes"`, and `extract_final_answer("just text")` is `"just text"`.

### What the checks test

The cell below the stubs runs these checks:

- the chain-of-thought prompt's content contains `step by step` (ignoring capital letters);
- extracting from `"Let's see... 12*13=156.\nAnswer: 156"` gives `"156"`;
- a reply with no marker comes back unchanged (`"no marker here"`).

If every check passes you will see `All checks passed.` printed. If a check fails, the message tells you what it expected and what your function returned. If you see `NotImplementedError`, replace the `TODO` and the `raise` line in that function with your code.

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    """Build a prompt that asks the question directly, wanting only a short answer.

    Args:
        question: the question text, for example "What is 12 * 13?".

    Returns:
        A list with ONE message dict, for example
        [{"role": "user", "content": "What is 12 * 13? Answer in as few words as possible."}]

    Steps:
        1. Write an instruction asking for the final answer only, in as few
           words as possible.
        2. Join the question and that instruction into one string.
        3. Return a one-item list holding {"role": "user", "content": <that string>}.
    """
    raise NotImplementedError("build_direct_prompt() is not implemented yet - replace the TODO with your code")


def build_cot_prompt(question: str) -> list[dict]:
    """Build a prompt that asks the model to reason step by step first.

    Args:
        question: the question text, for example "What is 12 * 13?".

    Returns:
        A list with ONE user message dict. Its content contains the question,
        the phrase "step by step", and a request to end the reply with a
        last line of the form 'Answer: <final answer>'.

    Steps:
        1. Start with the question text.
        2. Add an instruction to think step by step (use that exact phrase).
        3. Add an instruction to finish with a final line 'Answer: <your answer>'.
        4. Return a one-item list holding {"role": "user", "content": <the text>}.
    """
    raise NotImplementedError("build_cot_prompt() is not implemented yet - replace the TODO with your code")


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    """Pick the final answer out of a model reply.

    Args:
        reply_text: the whole reply, for example "Let's see... 12*13=156."
            followed by a new line and "Answer: 156".
        marker: the label that comes before the answer (default "Answer:").

    Returns:
        Everything after the LAST occurrence of `marker`, stripped of spaces
        and newlines, for example "156". If `marker` is not in the reply,
        the whole reply stripped, for example "no marker here".

    Steps:
        1. Check whether `marker` appears in `reply_text` (use `in`).
        2. If it does not, return `reply_text.strip()`.
        3. If it does, split at the last marker (rsplit with a limit of 1)
           and take the piece after it.
        4. Strip that piece and return it.
    """
    raise NotImplementedError("extract_final_answer() is not implemented yet - replace the TODO with your code")


assert "step by step" in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), 'expected ' + repr("step by step") + ' to be in build_cot_prompt("What is 12 * 13?")[0]["content"].lower(), but it is ' + repr(build_cot_prompt("What is 12 * 13?")[0]["content"].lower())
assert extract_final_answer("Let's see... 12*13=156.\nAnswer: 156") == "156", 'extract_final_answer("Let\'s see... 12*13=156.\\nAnswer: 156") should be ' + repr("156") + ", got " + repr(extract_final_answer("Let's see... 12*13=156.\nAnswer: 156"))
assert extract_final_answer("no marker here") == "no marker here", 'extract_final_answer("no marker here") should be ' + repr("no marker here") + ", got " + repr(extract_final_answer("no marker here"))

print("All checks passed.")

## Solution

In [ ]:
def build_direct_prompt(question: str) -> list[dict]:
    return [{"role": "user", "content": f"{question} Answer in as few words as possible."}]


def build_cot_prompt(question: str) -> list[dict]:
    return [{
        "role": "user",
        "content": f"{question}\n\nThink step by step, then end your reply with a final "
                   "line in exactly this form: 'Answer: <your answer>'",
    }]


def extract_final_answer(reply_text: str, marker: str = "Answer:") -> str:
    if marker not in reply_text:
        return reply_text.strip()
    return reply_text.rsplit(marker, 1)[1].strip()


print(build_cot_prompt("What is 12 * 13?"))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Compares direct-answer vs. chain-of-thought prompting on a handful of small reasoning questions - the kind small/fast models often get wrong when rushed into a one-shot answer.

In [ ]:
QUESTIONS = [
    ("If a train travels 60 miles in 45 minutes, what is its speed in miles per hour?", "80"),
    ("A shirt costs $40 after a 20% discount. What was the original price?", "50"),
    ("Sam has 3 times as many apples as Jo. Together they have 24 apples. How many does Sam have?", "18"),
]

for question, expected in QUESTIONS:
    direct_reply = call_groq(build_direct_prompt(question))
    cot_reply = call_groq(build_cot_prompt(question))
    cot_answer = extract_final_answer(cot_reply)

    print(f"Q: {question}")
    print(f"  expected:            {expected}")
    print(f"  direct answer:       {direct_reply.strip()[:60]}")
    print(f"  chain-of-thought:    {cot_answer}")
    print()